# Assignment 5: Exploratory Data Analysis & Pattern Mining (Satellite Dataset)

**Course Unit:** Unit VI – Exploratory Data Analysis Tools
**Student Name:** [Your Name]
**ID:** [Your ID]

---


## Part A – Data Loading & Initial EDA


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import IsolationForest

# 39. Load Satellite.csv
# Handling the non-standard semicolon-delimited triple-quoted CSV format
import io
try:
    # Attempting manual cleaning for robustness
    with open('Satellite.csv', 'r') as f:
        lines = f.readlines()
    data = [line.strip().replace('"', '').split(';') for line in lines]
    df = pd.DataFrame(data[1:], columns=data[0])
except FileNotFoundError:
    print("Please upload Satellite.csv to the Colab environment.")

# Clean numeric features
for col in df.columns:
    if col != 'classes':
        df[col] = pd.to_numeric(df[col], errors='coerce')

df = df.dropna(subset=['classes'])
df = df[df['classes'] != '']

print(f"Dataset Shape: {df.shape}")
print("\nData Types:")
print(df.dtypes.head())

class_dist = df['classes'].value_counts()
print("\nClass Distribution:")
print(class_dist)


### 40. Plot a pie chart and a bar chart of the class distribution


In [ ]:
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
class_dist.plot(kind='pie', autopct='%1.1f%%', startangle=140)
plt.title('Pie Chart of Class Distribution')

plt.subplot(1, 2, 2)
class_dist.plot(kind='bar')
plt.title('Bar Chart of Class Distribution')
plt.xlabel('Land-use Type')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()


### 41. Check for missing values


In [ ]:
print(f"Total Missing Values: {df.isnull().sum().sum()}")


## Part B – Clustering


In [ ]:
# 42. Scale the 36 spectral features
features = df.drop('classes', axis=1)
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)

# 43. Elbow Method
inertia = []
k_range = range(2, 11)
for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(features_scaled)
    inertia.append(kmeans.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(k_range, inertia, marker='o')
plt.title('Elbow Method for Optimal k')
plt.xlabel('Number of clusters (k)')
plt.ylabel('Inertia')
plt.grid(True)
plt.show()


In [ ]:
# 44. Fit K-Means with optimal k
optimal_k = 6
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
clusters = kmeans.fit_predict(features_scaled)

plt.figure(figsize=(8, 6))
plt.scatter(features_scaled[:, 0], features_scaled[:, 1], c=clusters, cmap='viridis', alpha=0.5)
plt.title(f'K-Means Clustering (k={optimal_k})')
plt.xlabel('Feature x1 (scaled)')
plt.ylabel('Feature x2 (scaled)')
plt.colorbar(label='Cluster')
plt.show()


In [ ]:
# 45. Silhouette Scores
def get_silhouette(k):
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(features_scaled)
    return silhouette_score(features_scaled, labels)

print(f"Silhouette Score (k={optimal_k}): {get_silhouette(optimal_k):.4f}")
print(f"Silhouette Score (k=3): {get_silhouette(3):.4f}")
print(f"Silhouette Score (k=7): {get_silhouette(7):.4f}")


In [ ]:
# 46. Cross-tabulation
cross_tab = pd.crosstab(df['classes'], clusters, rownames=['True'], colnames=['Cluster'])
print("\nCross-tabulation (True Labels vs Clusters):")
print(cross_tab)


## Part C – Classification & Outlier Detection


In [ ]:
# 47. Split and Train Decision Tree
X_train, X_test, y_train, y_test = train_test_split(features, df['classes'], test_size=0.20, stratify=df['classes'], random_state=42)
dt = DecisionTreeClassifier(max_depth=5, random_state=42)
dt.fit(X_train, y_train)
y_pred = dt.predict(X_test)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))


In [ ]:
# 48. Confusion Matrix heatmap
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=dt.classes_, yticklabels=dt.classes_)
plt.title('Confusion Matrix Heatmap')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()


In [ ]:
# 49. Isolation Forest
iso_forest = IsolationForest(contamination=0.05, random_state=42)
outliers = iso_forest.fit_predict(features)

plt.figure(figsize=(8, 6))
plt.scatter(features.iloc[:, 0], features.iloc[:, 1], c='blue', label='Normal', alpha=0.5)
plt.scatter(features.iloc[outliers == -1, 0], features.iloc[outliers == -1, 1], c='red', label='Anomaly')
plt.title(f"Outlier Detection: {(outliers == -1).sum()} anomalies detected")
plt.xlabel('Feature x1')
plt.ylabel('Feature x2')
plt.legend()
plt.show()


## Part D – Dashboard Logic (Tableau Simulations)

Below are the charts requested for the Tableau Dashboard.


In [ ]:
heatmap_data = df.groupby('classes')[['x.1', 'x.2']].mean()
plt.figure(figsize=(8, 6))
sns.heatmap(heatmap_data, annot=True, cmap='YlGnBu')
plt.title('Average x1 and x2 by Land-use Class')
plt.show()

dist_df = pd.DataFrame({
    'Type': ['True Classes'] * len(class_dist) + ['K-Means Clusters'] * optimal_k,
    'Label': list(class_dist.index) + [f'Cluster {i}' for i in range(optimal_k)],
    'Count': list(class_dist.values) + list(pd.Series(clusters).value_counts().sort_index().values)
})

plt.figure(figsize=(10, 6))
sns.barplot(data=dist_df, x='Label', y='Count', hue='Type')
plt.title('True Class vs K-Means Cluster Distribution')
plt.xticks(rotation=45)
plt.show()
